# DermRx Agent - Notebook 3: Molecular Intelligence with TxGemma & MedGemma 
## MedGemma Impact Challenge | Agentic Workflow Prize

**Purpose:** Validate TxGemmaf or molecular toxicity predictin and MedGemma for clinical reasoning - the two LLMs that power our medication safety pipeline. 
<br>
**Models:** [TxGemma 2B](https://huggingface.co/google/txgemma-2b-predict) (molecular property prediction), [MedGemma 4B](https://huggingface.co/google/medgemma-4b-it) (clinical reasoning and report synthesis)
  
<font color = "red"><u>**Key findings:**</u></font> TxGemma was NOT trained on drug-drug interaction datasets (DrugBank_DDI and TWOSIDES were explicity excluded). It missed fluconazole's well-known CPY2C9 inhibition - the mechanism behind the dangerous warfarin interaction. This finding shaped our entire architecture: DDInter handles DDI checking, TxGemma provides complementary molecular toxicity intelligence. 

<font color = "red"><u>**Technical Challenge:**</u></font> Fitting both TxGemma 2B and MedGemma 4B on a single T4 GPU (16GB VRAM) requires 4-bit quantization for both models. Together they use ~ 5-6GB, leaving room for MedSigLIP(~3GB) in the full pipeline. 

---

In Notebook 1 and 2, we built skin diagnosis (MedSigLIP) and drug safety data (DDInter + treatment table). Now we validate the two AI models that complete the pipeline - TxGemma for predicting molecular properties like toxicity and photosensitivity, and MedGemma for clinical reasoning and report synthesis.

## Setup 

Authenticate with HuggingFace and configure the environment. [TxGemma](https://huggingface.co/google/txgemma-2b-predict) and [MedGemma](https://huggingface.co/google/medgemma-4b-it) requires accepting the HAI-DEF terms of use. 

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

import torch 
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## Loading TxGemma 2B

[TxGemma](https://huggingface.co/google/txgemma-2b-predict) is Google's therapeutics prediction model, fine-tuned from Gemma 2 on the Therapeutics Data Commons (TDC) - 66 drug property prediction tasks. We use the 2B predict variant with 4-bit quantization to fit alongside MedGemma on a single T4. 

We also load the TDC prompt templates - these are the exact input formats TxGemma was trained on. 

In [ ]:
# download BitsAndBytesConfig
# !pip install --upgrade transformers accelerate bitsandbytes

In [2]:
import json 
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# loading the TDC prompt templates
tdc_prompts_filepath = hf_hub_download(
    repo_id = "google/txgemma-2b-predict",
    filename = "tdc_prompts.json",
)

with open(tdc_prompts_filepath, "r") as f:
    tdc_prompts = json.load(f)

print(f"TDC tasks available: {len(tdc_prompts)}")

tdc_prompts.json:   0%|          | 0.00/768k [00:00<?, ?B/s]

TDC tasks available: 703


In [3]:
# loading TxGemma 2B with 4-bit quantization
quantization_config = BitsAndBytesConfig(load_in_4bit=True)
predict_tokenizer = AutoTokenizer.from_pretrained("google/txgemma-2b-predict")
predict_model = AutoModelForCausalLM.from_pretrained(
    "google/txgemma-2b-predict",
    device_map = "auto",
    quantization_config = quantization_config
)

print(f"TxGemma 2B loaded and VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

TxGemma 2B loaded and VRAM used: 0.0 GB


## Exploring Relevant TDC Tasks

Out of 703 tasks, we need the ones relevant to medication safety for dermatology patients. We focus on six tasks that help us assess whether a treatment drug is safe to prescribe. 

In [5]:
# the 6 tasks relevant to our pipeline
RELEVANT_TASKS = {
    "Skin_Reaction": "Predicts photosensitivity/skin sensitization — critical for dermatology",
    "DILI": "Predicts drug-induced liver injury — safety for systemic treatments",
    "CYP2C9_Veith": "Predicts CYP2C9 inhibition — explains warfarin interactions",
    "CYP3A4_Veith": "Predicts CYP3A4 inhibition — major drug metabolism pathway",
    "hERG": "Predicts cardiac QT prolongation risk — safety screening",
    "ClinTox": "Predicts clinical trial toxicity — overall safety signal",
}

for task, description in RELEVANT_TASKS.items():
    if task in tdc_prompts:
        print(f"{task:15} - {description}")
    else:
        print(f"{task:15} - NOT FOUND")

print(tdc_prompts["Skin_Reaction"][:500])

Skin_Reaction   - Predicts photosensitivity/skin sensitization — critical for dermatology
DILI            - Predicts drug-induced liver injury — safety for systemic treatments
CYP2C9_Veith    - Predicts CYP2C9 inhibition — explains warfarin interactions
CYP3A4_Veith    - Predicts CYP3A4 inhibition — major drug metabolism pathway
hERG            - Predicts cardiac QT prolongation risk — safety screening
ClinTox         - Predicts clinical trial toxicity — overall safety signal
Instructions: Answer the following question about drug properties.
Context: Repetitive exposure to a chemical agent can induce an immune reaction in inherently susceptible individuals that leads to skin sensitization.
Question: Given a drug SMILES string, predict whether it
(A) does not cause a skin reaction (B) causes a skin reaction
Drug SMILES: {Drug SMILES}
Answer:


## Running TxGemma Predictions

We test TxGemma on key drugs from our pipeline - antifungals (the treatment candidates) and common patient medications (what we check interactions against). This validates whether TxGemma's molecular predictions are clinically meaningful.

In [6]:
def txgemma_predict(smiles, task):
    prompt = tdc_prompts[task].replace("{Drug SMILES}", smiles)
    input_ids = predict_tokenizer(prompt, return_tensors="pt").to(predict_model.device)
    outputs = predict_model.generate(**input_ids, max_new_tokens=8)
    response = predict_tokenizer.decode(
        outputs[0][len(input_ids["input_ids"][0]):], 
        skip_special_tokens=True
    ).strip()
    return response

# key drugs with SMILES for testing
test_drugs = {
    "fluconazole":  {"smiles": "OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F", "class": "antifungal"},
    "terbinafine":  {"smiles": "C(/C=C/c1ccccc1)(CN(C)C)CC#CC(C)(C)C", "class": "antifungal"},
    "clotrimazole": {"smiles": "ClC(c1ccccc1)(c1ccccc1)c1ccncc1", "class": "antifungal"},
    "ketoconazole": {"smiles": "O=C1N(CCOc2ccc(OC(c3ccc(Cl)cc3Cl)C3COC(Cn4ccnc4)(O3)C)cc2)CCCC1", "class": "antifungal"},
    "warfarin":     {"smiles": "CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O", "class": "patient_med"},
    "metformin":    {"smiles": "CN(C)C(=N)NC(=N)N", "class": "patient_med"},
    "isotretinoin": {"smiles": "CC1=CC(=O)C(C)=CC1=CC=CC(C)=CC=CC(C)=CC(=O)O", "class": "retinoid"},
}

In [7]:
tasks = [
    "Skin_Reaction",
    "DILI",
    "CYP2C9_Veith",
    "CYP3A4_Veith",
    "hERG",
    "ClinTox"    
]

# trying to create a table with header for our tasks
print(f"{'Drug':15} | {'Skin_Rxn':8} | {'DILI':8} | {'CYP2C9':8} | {'CYP3A4':8} | {'hERG':8} | {'ClinTox':8} | Class")
results_dict = {}
for drug_name, info in test_drugs.items():
    drug_results = {}
    row = []
    for task in tasks: 
        pred = txgemma_predict(info["smiles"], task)
        drug_results[task] = pred
        # flag = "FLAG" if "B" in pred, safe = "safe"
        label = "FLAG" if "B" in pred else "safe"
        row.append(label)
    results_dict[drug_name] = drug_results
    print(f"{drug_name:15} | {row[0]:8} | {row[1]:8} | {row[2]:8} | {row[3]:8} | {row[4]:8} | {row[5]:8} | {info['class']}")


Drug            | Skin_Rxn | DILI     | CYP2C9   | CYP3A4   | hERG     | ClinTox  | Class
fluconazole     | FLAG     | FLAG     | safe     | safe     | safe     | safe     | antifungal
terbinafine     | FLAG     | FLAG     | FLAG     | safe     | safe     | safe     | antifungal
clotrimazole    | FLAG     | FLAG     | FLAG     | FLAG     | FLAG     | safe     | antifungal
ketoconazole    | FLAG     | FLAG     | FLAG     | FLAG     | FLAG     | FLAG     | antifungal
warfarin        | FLAG     | FLAG     | FLAG     | safe     | safe     | safe     | patient_med
metformin       | FLAG     | safe     | safe     | safe     | safe     | safe     | patient_med
isotretinoin    | FLAG     | FLAG     | safe     | safe     | safe     | safe     | retinoid


## Critial Findings: TxGemma Misses Fluconazole CYP2C9

Fluconazole is a well-known CYP2C9 inhibitor - this is the exact mechanism that makes it dangerous with warfarin. But TxGemma predicted CYP2C0 = **safe** for fluconazole. Why? 

TxGemma was explicity **not trained** on drug-drug interaction datasets. DrugBank_DDI and TWOSIDES were excluded from its 66 TDC training tasks. CYP inhibition prediction (from the Veith dataset) tests whether a molecule inhibits an enzyme in isolation - it doesn't capture the clinical significance of that inhibition in the context of other drugs. 

**This shaped our entire architecture:**
- **DDInter** - primary source for drug-drug interactions (clinical, curated, 302K+ interactions)
- **TxGemma** - complementary molecule intelligence (toxicity, photosensitivity, CYP profiles)
- **Together** they form a complete safety picture that neither provides alone

This is why DermRX uses a multi-model approach rather than relying on any single model.

---

## Loading MedGemma 4B

[MedGemma](https://huggingface.co/google/medgemma-4b-it) is Google's medical multimodal language model. In our pipeline, it handles clinical reasoning - generating treatment recommendations and synthesizing safety findings into coherent clinical reports. We load it alongside TxGemma with 4-bit quantization.

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

medgemma_tokenizer = AutoTokenizer.from_pretrained("google/medgemma-4b-it")
medgemma_model = AutoModelForCausalLM.from_pretrained(
    "google/medgemma-4b-it",
    device_map = "auto",
    quantization_config = quantization_config
)

print(f"MedGemma 4B loaded and VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

MedGemma 4B loaded and VRAM used: 0.2 GB


## Testing MedGemma Clinical Reasoning 

MedGemma's primary role in our pipeline is **synthesizing clinical reports** - not selecting drugs. We discovered early on that MedGemma struggles to select from large drug list (it echoes them back). 

Let's first see how MedGemma handls a treatment question, then test its actual role - report synthesis. 

In [19]:
def medgemma_generate(prompt, max_tokens=256):
    encoded = medgemma_tokenizer(prompt, return_tensors="pt")
    input_len = len(encoded["input_ids"][0])
    inputs = {k: v.to(medgemma_model.device) for k, v in encoded.items()}
    
    outputs = medgemma_model.generate(
        **inputs, 
        max_new_tokens=max_tokens, 
        temperature=0.3, 
        do_sample=True,
        repetition_penalty=1.3,
    )
    response = medgemma_tokenizer.decode(
        outputs[0][input_len:], 
        skip_special_tokens=True
    ).strip()
    return response

In [20]:
# Test 1: Can MedGemma suggest treatments? 
prompt1 = """You are a clinical decision support system. A patient has been diagnosed with tinea corporis (fungal skin infection). 

Recommend the top 3 treatment options ranked by standard of care. For each, provide the drug name and brief rationale.

Keep your response concise."""

print(medgemma_generate(prompt1))

1.  **Topical Antifungal:** This is typically first-line for mild to moderate cases like Tinea Corporis due to its ease of use in treating localized infections at home without requiring medical supervision or prescription from an MD/DO as opposed to oral medication which requires it.. The most common topical antifungals used include clotrimazole cream, miconazole cream, terbinafine hydrochloride solution, etc., depending on local availability & physician preference; however all have similar efficacy against dermatophytes that cause ringworm such as *Trichophyton*, *Microsporum* ,and *Epidermophyton*. They work locally within the affected area killing fungal cells present there .

2. **Oral Antimycotic**: Oral medications may be needed if: Topical treatments fail after several weeks' consistent application OR when dealing with more extensive areas infected across multiple body parts where topical only can not reach effectively enough AND they also need faster results than what would occ

## Test 2: Clinical Report Synthesis (MedGemma's actual role)

In our pipeline, MedGemma doesn't pick drugs - the treatment table and agentic loop do that. MedGemma's job is to take all the findings (diagnosis, selected drug, rejected drugs, DDI results, TxGemma flags) and synthesize them into a coherent clinical report for the physician. 

In [21]:
# simulating the flagship warfarin scenario
synthesis_prompt = """You are a clinical decision support system generating a report for a primary care physician. 

DIAGNOSIS: Fungal Infection (Tinea Corporis)
Confidence: HIGH (42%)

PATIENT MEDICATIONS: warfarin, metformin, lisinopril

RECOMMENDED TREATMENT: terbinafine (topical)

REJECTED TREATMENTS: 
- fluconazole: REJECTED — Major DDI with warfarin (CYP2C9 inhibition increases warfarin levels, risk of bleeding)

SAFETY FINDINGS:
- terbinafine + warfarin: Moderate interaction — may slightly increase warfarin effect, monitor INR
- terbinafine molecular analysis: photosensitivity risk, hepatotoxicity risk

Generate a concise clinical report with:
1. CLINICAL SUMMARY (2-3 sentences)
2. TREATMENT RECOMMENDATION (selected drug with key guidance)
3. REASONING (why this drug was selected, what was avoided)
4. PATIENT EXPLANATION (simple language for the patient)"""

In [22]:
print(medgemma_generate(synthesis_prompt))

5. SAFETY INFORMATION/WARNINGS (key points to communicate about potential side effects and interactions).

---

**CLINICAL REPORT:**

**Clinical Summary:** The patient has been diagnosed with tinea corporis which is likely caused by dermatophytes such as *Trichophyton*, *Microsporum* or *Epidermophyton*. This fungal infection presents on their body in an itchy rash that can be red and scaly. Treatment will focus on topical application using antifungal medication like Terbinafine cream applied topically at night before showering; it should last no more than one week if used correctly according to instructions from your doctor's office..

**Treatment Recommendation:**  Terbinafine Topical Cream - Apply once nightly after bathing until symptoms resolve . Follow all directions provided when you pick up prescription!

**Reasoning:** Given the diagnosis confirmed via visual exam I recommend starting treatment immediately based upon guidelines established within medical literature regarding m

## Architecture Decision

From our testing across all three models, the roles are clear: 
1. **MedSigLIP** has Skin diagnosis role because of it's Zero-shot classification strength but it requires clinical photos not dermatoscopic images
2. **TxGemma 2B** has Molecular properties role because of it's Toxicity, photosensitivity, CYP profiles but it is not trained on DDI data - misses CYP2C9 for fluconazole
3. **MedGemma 4B** has Clinical reasoning role because of it Report synthesis and treatment rationale but it struggles with large drug selection, repetitive at times
4. **DDInter** has Drug Interactions role because of its 302K+ curated interactions, severity levels but it has no molecular-level predictions.

<font color = "red"><u>**Note**:</u></font> No single model can handle medication safety alone. The power is in combining them - each covers the other's blind spots. DDInter catches what TxGemma misses (drug interactions), TxGemma catches what DDInter can't predict (molecular toxicity), and MedGemma synthesizes it all into actionable clinical output. 

This multi-model orchestration is the core of DermRX Agent's agentic workflow. 

## Notebook 3 Summary

We've validated the two AI models that complete the DermRx pipeline: 
1. **TxGemma 2B** — Predicts molecular properties across 6 safety tasks. Ketoconazole flagged on all 6, metformin cleanest. Critical discovery: misses fluconazole CYP2C9.
2. **MedGemma 4B** — Generates treatment recommendations and clinical reports. Works well for synthesis, not reliable for drug selection from large lists.
3. **Architecture validated** — DDInter for DDI, TxGemma for molecular intelligence, MedGemma for clinical reasoning. Complementary, not redundant.

<font color = "red"><u>**Next**:</u></font> Notebook 4 brings everthing together - all 3 models loaded on a single GPU, running the complete agentic pipeline on the flagship warfarin scenario with real inferences. 